In [5]:
# =========================================
# 一体化实证代码：Dynamic FE + Quantile + SDM
# =========================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from linearmodels.panel import PanelOLS
import statsmodels.formula.api as smf

# ============== 0. 路径 ==============
data_file = "/Users/littlestars/Desktop/grain_project/data/processed/final_panel_regression_ready_with_zone.csv"
shp_file = "/Users/littlestars/Desktop/grain_project/data/large_files/gadm41_RUS_shp/gadm41_RUS_1.shp"   # 只有跑 SDM 时才需要

# ============== 1. 读取数据 ==============
df = pd.read_csv(data_file)
df = df.sort_values(["region", "year"]).reset_index(drop=True)

# 基础检查
required_cols = [
    "region", "year", "production", "area_total",
    "temp_grow_mean", "prec_grow_sum",
    "temp_annual_mean", "prec_annual_sum"
]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"缺少列: {c}")

# 对数变量
if "ln_production" not in df.columns:
    df = df[df["production"] > 0].copy()
    df["ln_production"] = np.log(df["production"])

if "ln_area" not in df.columns:
    df = df[df["area_total"] > 0].copy()
    df["ln_area"] = np.log(df["area_total"])

# 平方项
df["temp_grow_sq"] = df["temp_grow_mean"] ** 2
df["prec_grow_sq"] = df["prec_grow_sum"] ** 2

# 滞后项
df["ln_production_lag1"] = df.groupby("region")["ln_production"].shift(1)
df["temp_grow_lag1"] = df.groupby("region")["temp_grow_mean"].shift(1)
df["prec_grow_lag1"] = df.groupby("region")["prec_grow_sum"].shift(1)
df["temp_annual_lag1"] = df.groupby("region")["temp_annual_mean"].shift(1)
df["prec_annual_lag1"] = df.groupby("region")["prec_annual_sum"].shift(1)

# 区域分组，若没有就补一个 other
if "agro_zone" not in df.columns:
    df["agro_zone"] = "other"

# 设为 panel index
panel_df = df.set_index(["region", "year"]).copy()

print("数据 shape:", df.shape)
print("地区数:", df["region"].nunique())
print("年份范围:", df["year"].min(), "-", df["year"].max())

# =========================================
# 2. 动态 FE：气候当期 + 气候滞后
# lnY_it = b1 T_it + b2 P_it + b3 T_i,t-1 + b4 P_i,t-1 + FE + TE
# =========================================
reg1 = panel_df[[
    "ln_production",
    "temp_grow_mean", "prec_grow_sum",
    "temp_grow_lag1", "prec_grow_lag1"
]].dropna().copy()

m1 = PanelOLS(
    dependent=reg1["ln_production"],
    exog=reg1[["temp_grow_mean", "prec_grow_sum", "temp_grow_lag1", "prec_grow_lag1"]],
    entity_effects=True,
    time_effects=True
)
r1 = m1.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型1：动态 FE（气候滞后） ================\n")
print(r1.summary)

# =========================================
# 3. 动态 FE：加入被解释变量滞后
# lnY_it = phi lnY_i,t-1 + ...
# =========================================
reg2 = panel_df[[
    "ln_production",
    "ln_production_lag1",
    "temp_grow_mean", "prec_grow_sum",
    "temp_grow_lag1", "prec_grow_lag1"
]].dropna().copy()

m2 = PanelOLS(
    dependent=reg2["ln_production"],
    exog=reg2[[
        "ln_production_lag1",
        "temp_grow_mean", "prec_grow_sum",
        "temp_grow_lag1", "prec_grow_lag1"
    ]],
    entity_effects=True,
    time_effects=True
)
r2 = m2.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型2：动态 FE（含产量滞后） ================\n")
print(r2.summary)

# =========================================
# 4. 动态 FE：二次项 + 滞后
# =========================================
reg3 = panel_df[[
    "ln_production",
    "temp_grow_mean", "temp_grow_sq",
    "prec_grow_sum", "prec_grow_sq",
    "temp_grow_lag1", "prec_grow_lag1"
]].dropna().copy()

m3 = PanelOLS(
    dependent=reg3["ln_production"],
    exog=reg3[[
        "temp_grow_mean", "temp_grow_sq",
        "prec_grow_sum", "prec_grow_sq",
        "temp_grow_lag1", "prec_grow_lag1"
    ]],
    entity_effects=True,
    time_effects=True
)
r3 = m3.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型3：动态 FE（二次项 + 滞后） ================\n")
print(r3.summary)

# =========================================
# 5. 分位数回归：0.25 / 0.50 / 0.75
# 这里用地区与年份虚拟变量
# =========================================
qdf = df.dropna(subset=["ln_production", "temp_grow_mean", "prec_grow_sum"]).copy()

quantiles = [0.25, 0.50, 0.75]
q_results = {}

for q in quantiles:
    print(f"\n================ 分位数回归 q={q} ================\n")
    mod = smf.quantreg(
        "ln_production ~ temp_grow_mean + prec_grow_sum + C(region) + C(year)",
        data=qdf
    )
    res = mod.fit(q=q, max_iter=5000)
    q_results[q] = res
    print(res.summary())

# =========================================
# 6. 分位数回归：二次项版
# =========================================
qdf2 = df.dropna(subset=[
    "ln_production", "temp_grow_mean", "temp_grow_sq",
    "prec_grow_sum", "prec_grow_sq"
]).copy()

q2_results = {}

for q in quantiles:
    print(f"\n================ 分位数回归（二次项） q={q} ================\n")
    mod = smf.quantreg(
        "ln_production ~ temp_grow_mean + temp_grow_sq + prec_grow_sum + prec_grow_sq + C(region) + C(year)",
        data=qdf2
    )
    res = mod.fit(q=q, max_iter=5000)
    q2_results[q] = res
    print(res.summary())

# =========================================
# 7. 农业生态区交互：FE + interaction
# =========================================
df["agro_zone"] = df["agro_zone"].astype("category")
idf = df.dropna(subset=["ln_production", "temp_grow_mean", "prec_grow_sum", "agro_zone"]).copy()

m4 = smf.ols(
    "ln_production ~ temp_grow_mean * C(agro_zone) + prec_grow_sum * C(agro_zone) + C(region) + C(year)",
    data=idf
).fit(cov_type="HC1")

print("\n================ 模型4：农业生态区交互 ================\n")
print(m4.summary())

# =========================================
# 8. 提取核心结果表
# =========================================
core_rows = []

# PanelOLS
for model_name, res in [("dynamic_fe_lag", r1), ("dynamic_fe_y_lag", r2), ("dynamic_fe_quad", r3)]:
    for var in res.params.index:
        core_rows.append({
            "model": model_name,
            "variable": var,
            "coef": res.params[var],
            "t_or_z": res.tstats[var] if hasattr(res, "tstats") else np.nan,
            "pvalue": res.pvalues[var]
        })

# Quantile
for q, res in q_results.items():
    for var in ["temp_grow_mean", "prec_grow_sum"]:
        core_rows.append({
            "model": f"quantile_{q}",
            "variable": var,
            "coef": res.params.get(var, np.nan),
            "t_or_z": res.tvalues.get(var, np.nan),
            "pvalue": res.pvalues.get(var, np.nan)
        })

for q, res in q2_results.items():
    for var in ["temp_grow_mean", "temp_grow_sq", "prec_grow_sum", "prec_grow_sq"]:
        core_rows.append({
            "model": f"quantile_quad_{q}",
            "variable": var,
            "coef": res.params.get(var, np.nan),
            "t_or_z": res.tvalues.get(var, np.nan),
            "pvalue": res.pvalues.get(var, np.nan)
        })

# Interaction
for var in m4.params.index:
    if ("temp_grow_mean" in var) or ("prec_grow_sum" in var) or ("agro_zone" in var):
        core_rows.append({
            "model": "zone_interaction",
            "variable": var,
            "coef": m4.params[var],
            "t_or_z": m4.tvalues[var],
            "pvalue": m4.pvalues[var]
        })

core_table = pd.DataFrame(core_rows)
core_table.to_csv("/Users/littlestars/Desktop/grain_project/results/core_model_results.csv", index=False, encoding="utf-8-sig")
core_table.to_excel("/Users/littlestars/Desktop/grain_project/results/core_model_results.xlsx", index=False)

print("\n核心结果已保存：")
print("/Users/littlestars/Desktop/grain_project/results/core_model_results.csv")
print("/Users/littlestars/Desktop/grain_project/results/core_model_results.xlsx")

# =========================================
# 9. 可选：空间杜宾模型 SDM
# 需要 shp，并按 region 聚合成横截面（例如 2023 年）
# =========================================
try:
    import geopandas as gpd
    from libpysal.weights import Queen
    from spreg import ML_Lag

    # 取某一年做横截面
    target_year = 2023
    sdf = df[df["year"] == target_year].copy()

    # 读 shp
    gdf = gpd.read_file(shp_file).to_crs(epsg=4326)

    # 地区名列
    if "NL_NAME_1" in gdf.columns:
        gdf["region_std"] = gdf["NL_NAME_1"].astype(str).str.strip()
    elif "NAME_1" in gdf.columns:
        gdf["region_std"] = gdf["NAME_1"].astype(str).str.strip()
    else:
        raise ValueError("shp 中没有地区名称列")

    shp_fix = {
        "Чукотский АОк": "Чукотский автономный округ",
        "Ханты-Мансийский АОк": "Ханты-Мансийский автономный округ",
        "Ямало-Ненецкий АОк": "Ямало-Ненецкий автономный округ",
        "Камчатская край": "Камчатский край",
        "Карачаево-Черкессия Республика": "Карачаево-Черкесская Республика",
        "Республика Саха": "Республика Саха (Якутия)",
        "Республика Северная Осетия-Алани": "Республика Северная Осетия-Алания",
        "Чувашская Республика": "Чувашская Республика - Чувашия",
        "Санкт-Петербург (горсовет)": "г. Санкт-Петербург",
        "Кемеровская область": "Кемеровская область - Кузбасс",
        "Пермская край": "Пермский край",
        "Республика Чечено-Ингушская": "Чеченская Республика",
        "Респу́блика Ингуше́тия": "Республика Ингушетия",
    }
    gdf["region_std"] = gdf["region_std"].replace(shp_fix)

    if "region_std" not in sdf.columns:
        sdf["region_std"] = sdf["region"]

    xsec = gdf.merge(sdf, on="region_std", how="inner").copy()

    # 保留可用变量
    xsec = xsec.dropna(subset=["ln_production", "temp_grow_mean", "prec_grow_sum", "ln_area"]).copy()
    xsec = xsec.reset_index(drop=True)

    # 邻接权重
    w = Queen.from_dataframe(xsec)
    w.transform = "r"

    y = xsec[["ln_production"]].values
    X = xsec[["temp_grow_mean", "prec_grow_sum", "ln_area"]].values

    # 这里用 ML_Lag 作为基础空间滞后模型
    # 真正严格的 SDM 需要进一步手工加入 WX 或更细设定
    sdm_like = ML_Lag(
        y, X, w=w,
        name_y="ln_production",
        name_x=["temp_grow_mean", "prec_grow_sum", "ln_area"],
        name_w="queen_weights"
    )

    print("\n================ 空间模型（基础空间滞后版） ================\n")
    print(sdm_like.summary)

except Exception as e:
    print("\n空间模型部分未运行成功：", e)
    print("如果您现在主要是给老师汇报，前面的动态 FE + 分位数回归已经够用。")

数据 shape: (1170, 20)
地区数: 71
年份范围: 2007 - 2023

================ 模型1：动态 FE（气候滞后） ================

                          PanelOLS Estimation Summary                           
Dep. Variable:          ln_production   R-squared:                        0.0757
Estimator:                   PanelOLS   R-squared (Between):             -0.3125
No. Observations:                1099   R-squared (Within):               0.1012
Date:                Thu, Apr 02 2026   R-squared (Overall):             -0.3111
Time:                        14:42:58   Log-likelihood                   -202.69
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      20.654
Entities:                          71   P-value                           0.0000
Avg Obs:                       15.479   Distribution:                  F(4,1009)
Min Obs:                       4.0000                                           
Max Obs:  

In [5]:
# =========================================
# 结果自动输出：txt + csv + xlsx + png
# =========================================
import os

result_dir = "/Users/littlestars/Desktop/grain_project/results"
os.makedirs(result_dir, exist_ok=True)

# ---------- 1. 保存所有 summary 到 txt ----------
all_txt = os.path.join(result_dir, "all_model_results.txt")

with open(all_txt, "w", encoding="utf-8") as f:
    f.write("=============== 模型1：动态 FE（气候滞后） ===============\n")
    f.write(str(r1.summary))
    f.write("\n\n")

    f.write("=============== 模型2：动态 FE（含产量滞后） ===============\n")
    f.write(str(r2.summary))
    f.write("\n\n")

    f.write("=============== 模型3：动态 FE（二次项 + 滞后） ===============\n")
    f.write(str(r3.summary))
    f.write("\n\n")

    f.write("=============== 模型4：农业生态区交互 ===============\n")
    f.write(str(m4.summary()))
    f.write("\n\n")

    for q, res in q_results.items():
        f.write(f"=============== 分位数回归 q={q} ===============\n")
        f.write(str(res.summary()))
        f.write("\n\n")

    for q, res in q2_results.items():
        f.write(f"=============== 分位数回归（二次项） q={q} ===============\n")
        f.write(str(res.summary()))
        f.write("\n\n")

print("TXT 已保存：", all_txt)

# ---------- 2. 分别保存每个模型到独立 txt ----------
with open(os.path.join(result_dir, "model1_dynamic_fe_lag.txt"), "w", encoding="utf-8") as f:
    f.write(str(r1.summary))

with open(os.path.join(result_dir, "model2_dynamic_fe_y_lag.txt"), "w", encoding="utf-8") as f:
    f.write(str(r2.summary))

with open(os.path.join(result_dir, "model3_dynamic_fe_quad.txt"), "w", encoding="utf-8") as f:
    f.write(str(r3.summary))

with open(os.path.join(result_dir, "model4_zone_interaction.txt"), "w", encoding="utf-8") as f:
    f.write(str(m4.summary()))

for q, res in q_results.items():
    with open(os.path.join(result_dir, f"quantile_{str(q).replace('.', '_')}.txt"), "w", encoding="utf-8") as f:
        f.write(str(res.summary()))

for q, res in q2_results.items():
    with open(os.path.join(result_dir, f"quantile_quad_{str(q).replace('.', '_')}.txt"), "w", encoding="utf-8") as f:
        f.write(str(res.summary()))

print("各模型单独 TXT 已保存")

# ---------- 3. 核心结果表 ----------
core_rows = []

for model_name, res in [("dynamic_fe_lag", r1), ("dynamic_fe_y_lag", r2), ("dynamic_fe_quad", r3)]:
    for var in res.params.index:
        core_rows.append({
            "model": model_name,
            "variable": var,
            "coef": res.params[var],
            "t_or_z": res.tstats[var] if hasattr(res, "tstats") else None,
            "pvalue": res.pvalues[var]
        })

for q, res in q_results.items():
    for var in ["temp_grow_mean", "prec_grow_sum"]:
        core_rows.append({
            "model": f"quantile_{q}",
            "variable": var,
            "coef": res.params.get(var, None),
            "t_or_z": res.tvalues.get(var, None),
            "pvalue": res.pvalues.get(var, None)
        })

for q, res in q2_results.items():
    for var in ["temp_grow_mean", "temp_grow_sq", "prec_grow_sum", "prec_grow_sq"]:
        core_rows.append({
            "model": f"quantile_quad_{q}",
            "variable": var,
            "coef": res.params.get(var, None),
            "t_or_z": res.tvalues.get(var, None),
            "pvalue": res.pvalues.get(var, None)
        })

for var in m4.params.index:
    if ("temp_grow_mean" in var) or ("prec_grow_sum" in var) or ("agro_zone" in var):
        core_rows.append({
            "model": "zone_interaction",
            "variable": var,
            "coef": m4.params[var],
            "t_or_z": m4.tvalues[var],
            "pvalue": m4.pvalues[var]
        })

core_table = pd.DataFrame(core_rows)

core_csv = os.path.join(result_dir, "core_model_results.csv")
core_xlsx = os.path.join(result_dir, "core_model_results.xlsx")

core_table.to_csv(core_csv, index=False, encoding="utf-8-sig")
core_table.to_excel(core_xlsx, index=False)

print("核心结果表已保存：")
print(core_csv)
print(core_xlsx)

# ---------- 4. 如果空间模型成功，也存 txt ----------
try:
    with open(os.path.join(result_dir, "spatial_model_lag.txt"), "w", encoding="utf-8") as f:
        f.write(str(sdm_like.summary))
    print("空间模型 TXT 已保存")
except:
    print("空间模型未保存（可能未成功运行）")

TXT 已保存： /Users/littlestars/Desktop/grain_project/results/all_model_results.txt
各模型单独 TXT 已保存
核心结果表已保存：
/Users/littlestars/Desktop/grain_project/results/core_model_results.csv
/Users/littlestars/Desktop/grain_project/results/core_model_results.xlsx
空间模型 TXT 已保存


In [11]:
import pandas as pd
import numpy as np

# 路径改成你的
monthly_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_region_avg.csv"
out_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_with_spi.csv"

df = pd.read_csv(monthly_file)

# 基础整理
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
df["month"] = pd.to_numeric(df["month"], errors="coerce").astype(int)
df = df.sort_values(["region_std", "year", "month"]).reset_index(drop=True)

# 构造日期
df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2) + "-01"
)

# 3个月滚动降水
df["prec_3m"] = (
    df.groupby("region_std")["prec_mm"]
      .transform(lambda s: s.rolling(window=3, min_periods=3).sum())
)

# 对每个地区做标准化，近似 SPI-3
# 严格 SPI 通常要拟合 gamma 分布，这里先用 z-score 近似，够做作业/阶段性汇报
df["spi_3"] = (
    df.groupby("region_std")["prec_3m"]
      .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
)

df.to_csv(out_file, index=False, encoding="utf-8-sig")
print("已输出:", out_file)
print(df.head(12))

已输出: /Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_with_spi.csv
        region_std  year  month     temp_c     prec_mm       date     prec_3m  \
0   Алтайский край  2007      1  -8.037639   26.687159 2007-01-01         NaN   
1   Алтайский край  2007      2  -9.202633   43.776290 2007-02-01         NaN   
2   Алтайский край  2007      3  -7.837886   28.840900 2007-03-01   99.304349   
3   Алтайский край  2007      4   7.877717   23.689242 2007-04-01   96.306432   
4   Алтайский край  2007      5  13.257812  115.506936 2007-05-01  168.037078   
5   Алтайский край  2007      6  16.306906   91.380574 2007-06-01  230.576752   
6   Алтайский край  2007      7  21.630420   79.949462 2007-07-01  286.836972   
7   Алтайский край  2007      8  17.328133   50.939678 2007-08-01  222.269714   
8   Алтайский край  2007      9  13.398474   24.607635 2007-09-01  155.496775   
9   Алтайский край  2007     10   3.616601   46.709989 2007-10-01  122.257303   
10  Алтайский край 

In [13]:
import pandas as pd

spi_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_with_spi.csv"
out_year_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_yearly_with_spi.csv"

df = pd.read_csv(spi_file)

grow = df[df["month"].between(4, 9)].copy()

spi_year = (
    grow.groupby(["region_std", "year"], as_index=False)
        .agg(
            spi_grow_mean=("spi_3", "mean"),
            spi_grow_min=("spi_3", "min")
        )
)

spi_year.to_csv(out_year_file, index=False, encoding="utf-8-sig")
print("已输出:", out_year_file)
print(spi_year.head())

已输出: /Users/littlestars/Desktop/grain_project/data/processed/climate_yearly_with_spi.csv
       region_std  year  spi_grow_mean  spi_grow_min
0  Алтайский край  2007       1.049620     -0.838191
1  Алтайский край  2008       0.114225     -0.451164
2  Алтайский край  2009       0.882891     -0.834674
3  Алтайский край  2010      -0.047701     -0.792166
4  Алтайский край  2011      -0.286207     -0.980573


In [15]:
import pandas as pd

panel_file = "/Users/littlestars/Desktop/grain_project/data/processed/final_panel_regression_ready_with_zone.csv"
spi_year_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_yearly_with_spi.csv"
out_file = "/Users/littlestars/Desktop/grain_project/data/processed/final_panel_with_spi.csv"

panel = pd.read_csv(panel_file)
spi = pd.read_csv(spi_year_file)

panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)
spi["year"] = pd.to_numeric(spi["year"], errors="coerce").astype(int)

df = panel.merge(
    spi,
    on=["region_std", "year"],
    how="left"
)

df.to_csv(out_file, index=False, encoding="utf-8-sig")
print("已输出:", out_file)
print(df[["region", "year", "spi_grow_mean", "spi_grow_min"]].head())

已输出: /Users/littlestars/Desktop/grain_project/data/processed/final_panel_with_spi.csv
           region  year  spi_grow_mean  spi_grow_min
0  Алтайский край  2007       1.049620     -0.838191
1  Алтайский край  2008       0.114225     -0.451164
2  Алтайский край  2009       0.882891     -0.834674
3  Алтайский край  2010      -0.047701     -0.792166
4  Алтайский край  2011      -0.286207     -0.980573


In [17]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("/Users/littlestars/Desktop/grain_project/data/processed/final_panel_with_spi.csv")

reg_df = df.dropna(subset=["ln_production", "temp_grow_mean", "spi_grow_mean"]).copy()

m_spi = smf.ols(
    "ln_production ~ temp_grow_mean + spi_grow_mean + C(region) + C(year)",
    data=reg_df
).fit(cov_type="HC1")

print(m_spi.summary())

                            OLS Regression Results                            
Dep. Variable:          ln_production   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.978
Method:                 Least Squares   F-statistic:                     685.1
Date:                Fri, 03 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:41:12   Log-Likelihood:                -300.66
No. Observations:                1170   AIC:                             779.3
Df Residuals:                    1081   BIC:                             1230.
Df Model:                          88                                         
Covariance Type:                  HC1                                         
                                                     coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [19]:
reg_df2 = df.dropna(subset=["ln_production", "temp_grow_mean", "spi_grow_min"]).copy()

m_spi_min = smf.ols(
    "ln_production ~ temp_grow_mean + spi_grow_min + C(region) + C(year)",
    data=reg_df2
).fit(cov_type="HC1")

print(m_spi_min.summary())

                            OLS Regression Results                            
Dep. Variable:          ln_production   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.977
Method:                 Least Squares   F-statistic:                     663.2
Date:                Fri, 03 Apr 2026   Prob (F-statistic):               0.00
Time:                        19:44:31   Log-Likelihood:                -308.24
No. Observations:                1170   AIC:                             794.5
Df Residuals:                    1081   BIC:                             1245.
Df Model:                          88                                         
Covariance Type:                  HC1                                         
                                                     coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [21]:
import pandas as pd
import numpy as np

monthly_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_region_avg.csv"
out_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_with_spei_like.csv"

df = pd.read_csv(monthly_file)
df = df.sort_values(["region_std", "year", "month"]).reset_index(drop=True)

# 简化 PET 代理：温度越高，蒸散越强
# 这里只做近似，不是正式 Thornthwaite / Penman-Monteith
df["pet_proxy"] = np.maximum(df["temp_c"], 0) * 5.0

# 水分平衡
df["water_balance"] = df["prec_mm"] - df["pet_proxy"]

# 3个月滚动水分平衡
df["wb_3m"] = (
    df.groupby("region_std")["water_balance"]
      .transform(lambda s: s.rolling(window=3, min_periods=3).sum())
)

# 标准化，得到简化版 SPEI
df["spei_like_3"] = (
    df.groupby("region_std")["wb_3m"]
      .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
)

df.to_csv(out_file, index=False, encoding="utf-8-sig")
print("已输出:", out_file)
print(df.head(12))

已输出: /Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_with_spei_like.csv
        region_std  year  month     temp_c     prec_mm   pet_proxy  \
0   Алтайский край  2007      1  -8.037639   26.687159    0.000000   
1   Алтайский край  2007      2  -9.202633   43.776290    0.000000   
2   Алтайский край  2007      3  -7.837886   28.840900    0.000000   
3   Алтайский край  2007      4   7.877717   23.689242   39.388582   
4   Алтайский край  2007      5  13.257812  115.506936   66.289062   
5   Алтайский край  2007      6  16.306906   91.380574   81.534530   
6   Алтайский край  2007      7  21.630420   79.949462  108.152100   
7   Алтайский край  2007      8  17.328133   50.939678   86.640665   
8   Алтайский край  2007      9  13.398474   24.607635   66.992370   
9   Алтайский край  2007     10   3.616601   46.709989   18.083006   
10  Алтайский край  2007     11  -5.162679   41.096094    0.000000   
11  Алтайский край  2007     12 -10.468814   33.680097    0.0000

In [23]:
# =========================================
# SPI + Dynamic FE + Quantile Regression
# =========================================

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

# =========================================
# 0. 路径
# =========================================
monthly_file = "/Users/littlestars/Desktop/grain_project/data/processed/climate_monthly_region_avg.csv"
panel_file   = "/Users/littlestars/Desktop/grain_project/data/processed/final_panel_regression_ready_with_zone.csv"
result_dir   = "/Users/littlestars/Desktop/grain_project/results"
os.makedirs(result_dir, exist_ok=True)

# =========================================
# 1. 构造 SPI-3
# =========================================
monthly = pd.read_csv(monthly_file)

monthly["year"] = pd.to_numeric(monthly["year"], errors="coerce").astype(int)
monthly["month"] = pd.to_numeric(monthly["month"], errors="coerce").astype(int)
monthly = monthly.sort_values(["region_std", "year", "month"]).reset_index(drop=True)

monthly["date"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2) + "-01"
)

# 3个月滚动降水
monthly["prec_3m"] = (
    monthly.groupby("region_std")["prec_mm"]
    .transform(lambda s: s.rolling(window=3, min_periods=3).sum())
)

# 近似 SPI-3：按地区 z-score 标准化
monthly["spi_3"] = (
    monthly.groupby("region_std")["prec_3m"]
    .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
)

# 生长季聚合（4-9月）
grow = monthly[monthly["month"].between(4, 9)].copy()

spi_year = (
    grow.groupby(["region_std", "year"], as_index=False)
    .agg(
        spi_grow_mean=("spi_3", "mean"),
        spi_grow_min=("spi_3", "min")
    )
)

spi_year.to_csv(os.path.join(result_dir, "climate_yearly_with_spi.csv"), index=False, encoding="utf-8-sig")

# =========================================
# 2. 合并进最终回归表
# =========================================
df = pd.read_csv(panel_file)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
spi_year["year"] = pd.to_numeric(spi_year["year"], errors="coerce").astype(int)

df = df.merge(
    spi_year,
    on=["region_std", "year"],
    how="left"
)

# 如果缺 ln_production / ln_area 就补
if "ln_production" not in df.columns:
    df = df[df["production"] > 0].copy()
    df["ln_production"] = np.log(df["production"])

if "ln_area" not in df.columns:
    df = df[df["area_total"] > 0].copy()
    df["ln_area"] = np.log(df["area_total"])

# 构造平方项
df["temp_grow_sq"] = df["temp_grow_mean"] ** 2
df["spi_grow_mean_sq"] = df["spi_grow_mean"] ** 2

# 构造滞后项
df = df.sort_values(["region", "year"]).reset_index(drop=True)
df["ln_production_lag1"] = df.groupby("region")["ln_production"].shift(1)
df["temp_grow_lag1"] = df.groupby("region")["temp_grow_mean"].shift(1)
df["spi_grow_mean_lag1"] = df.groupby("region")["spi_grow_mean"].shift(1)
df["spi_grow_min_lag1"] = df.groupby("region")["spi_grow_min"].shift(1)

# 保存合并后数据
df.to_csv(os.path.join(result_dir, "final_panel_with_spi.csv"), index=False, encoding="utf-8-sig")
df.to_excel(os.path.join(result_dir, "final_panel_with_spi.xlsx"), index=False)

print("数据已合并完成")
print(df[["region", "year", "temp_grow_mean", "spi_grow_mean", "spi_grow_min"]].head())

# =========================================
# 3. Dynamic FE：用 SPI 替代降水
# =========================================
panel_df = df.set_index(["region", "year"]).copy()

# 模型1：当期 SPI + 滞后 SPI
reg1 = panel_df[[
    "ln_production",
    "temp_grow_mean", "spi_grow_mean",
    "temp_grow_lag1", "spi_grow_mean_lag1"
]].dropna().copy()

m1 = PanelOLS(
    dependent=reg1["ln_production"],
    exog=reg1[[
        "temp_grow_mean", "spi_grow_mean",
        "temp_grow_lag1", "spi_grow_mean_lag1"
    ]],
    entity_effects=True,
    time_effects=True
)
r1 = m1.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型1：Dynamic FE + SPI ================\n")
print(r1.summary)

# 模型2：加被解释变量滞后
reg2 = panel_df[[
    "ln_production",
    "ln_production_lag1",
    "temp_grow_mean", "spi_grow_mean",
    "temp_grow_lag1", "spi_grow_mean_lag1"
]].dropna().copy()

m2 = PanelOLS(
    dependent=reg2["ln_production"],
    exog=reg2[[
        "ln_production_lag1",
        "temp_grow_mean", "spi_grow_mean",
        "temp_grow_lag1", "spi_grow_mean_lag1"
    ]],
    entity_effects=True,
    time_effects=True
)
r2 = m2.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型2：Dynamic FE + SPI + lag(y) ================\n")
print(r2.summary)

# 模型3：二次项
reg3 = panel_df[[
    "ln_production",
    "temp_grow_mean", "temp_grow_sq",
    "spi_grow_mean", "spi_grow_mean_sq",
    "temp_grow_lag1", "spi_grow_mean_lag1"
]].dropna().copy()

m3 = PanelOLS(
    dependent=reg3["ln_production"],
    exog=reg3[[
        "temp_grow_mean", "temp_grow_sq",
        "spi_grow_mean", "spi_grow_mean_sq",
        "temp_grow_lag1", "spi_grow_mean_lag1"
    ]],
    entity_effects=True,
    time_effects=True
)
r3 = m3.fit(cov_type="clustered", cluster_entity=True)

print("\n================ 模型3：Dynamic FE + SPI（二次项） ================\n")
print(r3.summary)

# =========================================
# 4. 分位数回归：用 SPI
# =========================================
qdf = df.dropna(subset=["ln_production", "temp_grow_mean", "spi_grow_mean"]).copy()
quantiles = [0.25, 0.50, 0.75]
q_results = {}

for q in quantiles:
    print(f"\n================ 分位数回归 q={q}（SPI） ================\n")
    mod = smf.quantreg(
        "ln_production ~ temp_grow_mean + spi_grow_mean + C(region) + C(year)",
        data=qdf
    )
    res = mod.fit(q=q, max_iter=5000)
    q_results[q] = res
    print(res.summary())

# 二次项分位数回归
qdf2 = df.dropna(subset=[
    "ln_production", "temp_grow_mean", "temp_grow_sq",
    "spi_grow_mean", "spi_grow_mean_sq"
]).copy()

q2_results = {}

for q in quantiles:
    print(f"\n================ 分位数回归（二次项） q={q}（SPI） ================\n")
    mod = smf.quantreg(
        "ln_production ~ temp_grow_mean + temp_grow_sq + spi_grow_mean + spi_grow_mean_sq + C(region) + C(year)",
        data=qdf2
    )
    res = mod.fit(q=q, max_iter=5000)
    q2_results[q] = res
    print(res.summary())

# =========================================
# 5. 保存结果到 txt
# =========================================
out_txt = os.path.join(result_dir, "spi_model_results.txt")

with open(out_txt, "w", encoding="utf-8") as f:
    f.write("=============== 模型1：Dynamic FE + SPI ===============\n")
    f.write(str(r1.summary))
    f.write("\n\n")

    f.write("=============== 模型2：Dynamic FE + SPI + lag(y) ===============\n")
    f.write(str(r2.summary))
    f.write("\n\n")

    f.write("=============== 模型3：Dynamic FE + SPI（二次项） ===============\n")
    f.write(str(r3.summary))
    f.write("\n\n")

    for q, res in q_results.items():
        f.write(f"=============== 分位数回归 q={q}（SPI） ===============\n")
        f.write(str(res.summary()))
        f.write("\n\n")

    for q, res in q2_results.items():
        f.write(f"=============== 分位数回归（二次项） q={q}（SPI） ===============\n")
        f.write(str(res.summary()))
        f.write("\n\n")

print("\n结果已保存到：")
print(out_txt)

# =========================================
# 6. 提取核心系数表
# =========================================
core_rows = []

for model_name, res in [("dynamic_fe_spi", r1), ("dynamic_fe_spi_lagy", r2), ("dynamic_fe_spi_quad", r3)]:
    for var in res.params.index:
        core_rows.append({
            "model": model_name,
            "variable": var,
            "coef": res.params[var],
            "t_or_z": res.tstats[var] if hasattr(res, "tstats") else np.nan,
            "pvalue": res.pvalues[var]
        })

for q, res in q_results.items():
    for var in ["temp_grow_mean", "spi_grow_mean"]:
        core_rows.append({
            "model": f"quantile_spi_{q}",
            "variable": var,
            "coef": res.params.get(var, np.nan),
            "t_or_z": res.tvalues.get(var, np.nan),
            "pvalue": res.pvalues.get(var, np.nan)
        })

for q, res in q2_results.items():
    for var in ["temp_grow_mean", "temp_grow_sq", "spi_grow_mean", "spi_grow_mean_sq"]:
        core_rows.append({
            "model": f"quantile_spi_quad_{q}",
            "variable": var,
            "coef": res.params.get(var, np.nan),
            "t_or_z": res.tvalues.get(var, np.nan),
            "pvalue": res.pvalues.get(var, np.nan)
        })

core_table = pd.DataFrame(core_rows)
core_table.to_csv(os.path.join(result_dir, "spi_core_results.csv"), index=False, encoding="utf-8-sig")
core_table.to_excel(os.path.join(result_dir, "spi_core_results.xlsx"), index=False)

print("核心结果表已保存")

数据已合并完成
           region  year  temp_grow_mean  spi_grow_mean  spi_grow_min
0  Алтайский край  2007       14.966577       1.049620     -0.838191
1  Алтайский край  2008       14.458340       0.114225     -0.451164
2  Алтайский край  2009       13.675152       0.882891     -0.834674
3  Алтайский край  2010       13.087710      -0.047701     -0.792166
4  Алтайский край  2011       14.869391      -0.286207     -0.980573

================ 模型1：Dynamic FE + SPI ================

                          PanelOLS Estimation Summary                           
Dep. Variable:          ln_production   R-squared:                        0.0790
Estimator:                   PanelOLS   R-squared (Between):             -0.2191
No. Observations:                1099   R-squared (Within):               0.0731
Date:                Fri, Apr 03 2026   R-squared (Overall):             -0.2181
Time:                        19:46:32   Log-likelihood                   -200.69
Cov. Estimator:             Cluster